In [12]:
import pandas as pd
import numpy as np
import re
import hashlib
from pathlib import Path

# Helper Functions

In [13]:
def stable_id(*parts: Any, prefix: str = '') -> str:
    raw = '||'.join('' if p is None else str(p) for p in parts)
    h = hashlib.md5(raw.encode('utf-8')).hexdigest()[:12]
    return f'{prefix}{h}' if prefix else h

def normalize_system_category(category: Any, text: str) -> str:
    c = category if pd.notna(category) else ''
    c_l = c.lower()
    if c_l in CATEGORY_MAP:
        return CATEGORY_MAP[c_l]
    t = text.lower()
    for label, kws in PART_KEYWORDS.items():
        if any(k in t for k in kws):
            if label in ['turbo', 'intake']:
                return 'engine_power'
            return label
    return 'other'
    
def safe_literal_list(x: Any) -> list:
    if pd.isna(x) or x in ('', '[]'):
        return []
    try:
        val = ast.literal_eval(str(x))
        return val if isinstance(val, list) else [val]
    except Exception:
        return []


def first_image(x: Any) -> str | None:
    urls = safe_literal_list(x)
    for url in urls:
        u = str(url)
        if any(ext in u.lower() for ext in ['.jpg', '.jpeg', '.png', '.webp']):
            return u
    return None

def extract_subsystem(text: str) -> str:
    t = text.lower()
    for label, kws in PART_KEYWORDS.items():
        if any(k in t for k in kws):
            return label
    return 'other'

def extract_fitment_flags(text: str) -> dict[str, Any]:
    t = text.lower()
    years = [int(y) for y in re.findall(r'\b(20[12][0-9]|2030)\b', t)]
    
    # Filter years to valid A90 Supra range (2019-2026)
    valid_years = [y for y in years if 2019 <= y <= 2026]
    
    # Use A90 Supra default range if no valid years found
    if valid_years:
        model_year_min = min(valid_years)
        model_year_max = max(valid_years)
    else:
        model_year_min = 2019
        model_year_max = 2026

    car_fitment_flags = {
        'fits_gr_supra': bool(re.search(r'gr supra|mkv supra|mk5 supra|j29|a90|a91', t)),
        'fits_a90': 'a90' in t or 'j29' in t,
        'fits_a91': 'a91' in t,
        'fits_b58': 'b58' in t or '3.0' in t,
        'fits_b48': 'b48' in t or '2.0' in t,
        'fits_3_0': 'b58' in t or '3.0' in t,
        'fits_2_0': 'b48' in t or '2.0' in t,
        'model_year_min': model_year_min,
        'model_year_max': model_year_max,
    }
    
    return car_fitment_flags

def extract_numeric_claims(text: str) -> dict[str, Any]:
    t = text.lower()
    hp_vals = [int(v.replace('+', '')) for v, _ in re.findall(r'(\+?\d{1,4})\s?(whp|hp|horsepower)\b', t)]
    tq_vals = [int(v.replace('+', '')) for v, _ in re.findall(r'(\+?\d{1,4})\s?(tq|torque|lb[- ]?ft|ft[- ]?lb)\b', t)]
    weight_matches = re.findall(r'([+-]?\d+(?:\.\d+)?)\s?(lbs?|pounds|kg)\b', t)
    weights_lbs = []
    for val, unit in weight_matches:
        f = float(val)
        if unit == 'kg':
            f *= 2.20462
        weights_lbs.append(f)


    extracted_numeric_claims = {
        'claimed_hp_gain_min': min(hp_vals) if hp_vals else None,
        'claimed_hp_gain_max': max(hp_vals) if hp_vals else None,
        'claimed_tq_gain_min': min(tq_vals) if tq_vals else None,
        'claimed_tq_gain_max': max(tq_vals) if tq_vals else None,
        'weight_delta_lbs': weights_lbs[0] if weights_lbs else None,
        'hp_gain_source': 'stated' if hp_vals else 'missing',
    }

    return 


In [14]:
#risk_dependencies analyzes product text, category, and subsystem to infer installation and risk attributes.
def risk_dependencies(text: str, system_category: str, subsystem: str) -> dict[str, Any]:
    t = text.lower()
    requires_tune = any(keyword in text for keyword in ['requires tune', 'tune required', 'ecu tune', 'calibration required']) or subsystem in ['downpipe', 'turbo', 'fueling']
    requires_fueling = any(keyword in text for keyword in ['e85', 'ethanol', 'port injection', 'fuel pump', 'injector']) or subsystem == 'turbo'
    requires_cooling = any(keyword in text for keyword in ['heat exchanger', 'oil cooler', 'radiator', 'cooling']) or subsystem == 'turbo'
    requires_install = subsystem in ['turbo', 'fueling', 'suspension', 'brakes'] or any(keyword in text for keyword in ['professional installation', 'install by a professional'])
    
    if any(keyword in text for keyword in ['catless', 'off-road', 'race use only', 'not street legal']):
        emissions_risk = 'high'
    elif subsystem in ['downpipe', 'exhaust', 'tune'] or system_category in ['exhaust', 'tuning']:
        emissions_risk = 'medium'
    else:
        emissions_risk = 'low'
    if subsystem in ['turbo', 'fueling']:
        reliability_risk = 'high'
    elif subsystem in ['tune', 'downpipe', 'suspension', 'brakes']:
        reliability_risk = 'medium'
    else:
        reliability_risk = 'low'
    if subsystem in ['turbo', 'fueling']:
        install_complexity = 'high'
    elif subsystem in ['suspension', 'brakes', 'aero', 'downpipe']:
        install_complexity = 'medium'
    else:
        install_complexity = 'low'


    risk_dependencies ={
        'requires_tune': requires_tune,
        'requires_fueling': requires_fueling,
        'requires_cooling': requires_cooling,
        'requires_professional_install': requires_install,
        'emissions_risk': emissions_risk,
        'reliability_risk': reliability_risk,
        'install_complexity': install_complexity,
    }
    return risk_dependencies

In [15]:
CATEGORY_MAP = {
    'brake rotors': 'brakes', 'brakes': 'brakes', 'brake pads': 'brakes',
    'intakes': 'engine_power', 'intake': 'engine_power', 'intake filters': 'engine_power',
    'intake manifolds': 'engine_power', 'performance': 'engine_power',
    'performance accessories': 'engine_power', 'turbo': 'engine_power', 'turbos': 'engine_power',
    'turbo kits': 'engine_power', 'turbo kit': 'engine_power', 'bov': 'engine_power',
    'camshafts': 'engine_power', 'charge/boost pipes': 'engine_power',
    'downpipes': 'exhaust', 'exhaust': 'exhaust', 'exhausts': 'exhaust',
    'tuning': 'tuning', 'ecu': 'tuning', 'fueling': 'fueling',
    'port injection kit': 'fueling', 'flex fuel': 'fueling', 'cooling': 'cooling',
    'heat exchangers': 'cooling', 'suspension': 'suspension', 'coilovers': 'suspension',
    'springs': 'suspension', 'suspension components': 'suspension',
    'aero': 'aero', 'front lips/splitters': 'aero', 'spoilers & wings': 'aero',
    'rear skirts/spats': 'aero', 'side skirts': 'aero', 'wheels': 'wheels_tires',
    'tires': 'wheels_tires', 'maintenance': 'maintenance', 'electronics': 'electronics',
    'styling': 'styling', 'engine dress up': 'styling', 'body kits': 'styling',
    'mirrors': 'styling', 'steering wheel': 'styling', 'accessories': 'accessories',
}


PART_KEYWORDS = {
    'turbo': ['turbo', 'pure700', 'pure 700', 'g35', 'g42', 'precision', 'hybrid turbo'],
    'tune': ['tune', 'bootmod3', 'bm3', 'ecutek', 'mhd', 'motec', 'femto', 'ecu unlock'],
    'downpipe': ['downpipe', 'catless', 'catted dp', 'high-flow cat'],
    'intake': ['intake', 'airbox', 'snorkel', 'charge pipe', 'inlet'],
    'fueling': ['fuel pump', 'hpfp', 'lpfp', 'port injection', 'injector', 'e50', 'e85', 'ethanol'],
    'cooling': ['heat exchanger', 'radiator', 'oil cooler', 'cooling', 'intercooler'],
    'exhaust': ['exhaust', 'catback', 'straight pipe', 'muffler', 'titanium exhaust'],
    'brakes': ['brake', 'bbk', 'rotor', 'pad', 'caliper', 'alcon', 'brembo'],
    'suspension': ['coilover', 'spring', 'sway bar', 'control arm', 'camber', 'spl', 'mcs'],
    'aero': ['wing', 'splitter', 'diffuser', 'canard', 'verus', 'apr'],
    'wheels_tires': ['tire', 'tyre', 'wheel', 'apex', 'volk', 'te37', 'nankang', 'yokohama'],
}

KNOWN_TRACKS = [
    'lime rock', 'palmer motorsports', 'buttonwillow', 'road atlanta', 'vir', 'watkins glen',
    'laguna seca', 'summit point', 'cota', 'circuit of the americas', 'sebring', 'daytona',
    'thunderhill', 'willow springs', 'autobahn', 'njmp', 'mid-ohio', 'sonoma'
]

In [16]:
def clean_text(x: Any):
    if pd.isna(x):
        return None
    string = str(x)
    #removing html tags and non-breaking spaces, normalizing whitespace
    string = re.sub(r'<[^>]+>', ' ', string)
    string = string.replace('\xa0', ' ')
    string = re.sub(r'\s+', ' ', string).strip()

    return string or None


def parse_lap_time_seconds(text: str) -> float | None:
    # catches formats like 57.109, 1:48.30, 2:01.123; avoids dates by context-limiting in lap posts only
    candidates = re.findall(r'(?<!\d)(?:(\d{1,2}):)?(\d{1,2})\.(\d{2,3})(?!\d)', text)
    if not candidates:
        return None
    vals = []
    for mins, sec, frac in candidates:
        s = int(sec) + int(frac) / (1000 if len(frac) == 3 else 100)
        if mins:
            s += int(mins) * 60
        if 30 <= s <= 300:
            vals.append(s)
    return min(vals) if vals else None


def extract_track(text: str) -> str | None:
    t = text.lower()
    for tr in KNOWN_TRACKS:
        if tr in t:
            return tr.title()
    return None


def extract_part_mentions(text: str) -> list[str]:
    t = text.lower()
    mentions = []
    for label, kws in PART_KEYWORDS.items():
        if any(k in t for k in kws):
            mentions.append(label)
    return sorted(set(mentions))

# Fourm data

In [17]:
RAW_DIR = Path('../backend/app/data')
OUT_DIR = RAW_DIR / 'processed/redlineiq_cleaned'
OUT_DIR.mkdir(exist_ok=True)

PATHS = {
    'car_dyno': RAW_DIR / 'ingestion/raw/forum_data_raw/forum_data/car_dyno_csv.csv',
    'exhaust_poll': RAW_DIR / 'ingestion/raw/forum_data_raw/forum_data/exhaust_poll_csv.csv',
    'lap_times': RAW_DIR / 'ingestion/raw/forum_data_raw/forum_data/lap_times_csv.csv',
    'power_guide': RAW_DIR / 'ingestion/raw/forum_data_raw/forum_data/power_guide_csv.csv',
    'race_track': RAW_DIR / 'ingestion/raw/forum_data_raw/forum_data/race_track_csv.csv',
    'street_build_forum': RAW_DIR / 'ingestion/raw/forum_data_raw/forum_data/street_build_forum_csv.csv',
    'time_attack_build': RAW_DIR / 'ingestion/raw/forum_data_raw/forum_data/time_attack_build_csv.csv',
}

FORUM_SOURCES = [
    'car_dyno', 'exhaust_poll', 'lap_times', 'power_guide', 'race_track',
    'street_build_forum', 'time_attack_build'
]


# ETL Pipeline

In [18]:
forum_dataframes = []
for src in FORUM_SOURCES:
    path = PATHS.get(src)
    if path is None or not path.exists():
        print(f"Skipping {src}: file not found at {path}")
        continue
    df = pd.read_csv(path)
    df['source_dataset'] = src
    forum_dataframes.append(df)

In [19]:
forum_df = pd.concat(forum_dataframes, ignore_index=True)
forum_df['cleaned_content'] = forum_df['content'].apply(clean_text)

forum_df

,author,content,date,post_id,links,images,youtube_videos,source_dataset,cleaned_content
0,BrettS,339hp and 427ft lb torque\nhttps://www.carandd...,2019-05-21T11:02:14-0700,post-42418,['https://www.caranddriver.com/news/a27543113/...,['https://hips.hearstapps.com/hmg-prod.s3.amaz...,[],car_dyno,339hp and 427ft lb torque https://www.caranddr...
1,Guff,NaN,2019-05-21T11:07:31-0700,post-42420,[],[],[],car_dyno,NaN
2,Supra Turbo,Impressive. Can't wait to hear more about the ...,2019-05-21T11:11:58-0700,post-42421,[],[],[],car_dyno,Impressive. Can't wait to hear more about the ...
3,justbake,At 15% drivetrain loss thats 398hp and 503 lb-...,2019-05-21T11:19:26-0700,post-42423,[],[],[],car_dyno,At 15% drivetrain loss thats 398hp and 503 lb-...
4,A70TTR,"""it's so underpowered"" though",2019-05-21T11:54:42-0700,post-42431,[],[],[],car_dyno,"""it's so underpowered"" though"
...,...,...,...,...,...,...,...,...,...
7160,TBK,General updates:\nTime to renew the registrati...,2026-01-31T22:54:19-0800,post-543464,[],['https://cdn.supramkv.com/attachments/151/151...,[],time_attack_build,General updates: Time to renew the registratio...
7161,The Ginga Ninja,TBK said:\nAnybody have an experience working ...,2026-02-06T12:32:15-0800,post-545648,['https://www.supramkv.com/goto/post?id=543464'],[],[],time_attack_build,TBK said: Anybody have an experience working w...
7162,Rensuhlo,TBK said:\nClick to expand...\nLooks...familia...,2026-02-06T13:34:37-0800,post-545666,['https://www.supramkv.com/goto/post?id=543464'],['https://cdn.supramkv.com/attachments/151/151...,[],time_attack_build,TBK said: Click to expand... Looks...familiar lol
7163,TBK,The Ginga Ninja said:\ni have worked with Ryan...,2026-02-07T06:06:33-0800,post-545841,['https://www.supramkv.com/goto/post?id=545648...,['https://cdn.supramkv.com/attachments/151/151...,[],time_attack_build,The Ginga Ninja said: i have worked with Ryan ...


In [20]:
forum_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7165 entries, 0 to 7164
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   author           7165 non-null   str  
 1   content          7140 non-null   str  
 2   date             7165 non-null   str  
 3   post_id          7165 non-null   str  
 4   links            7165 non-null   str  
 5   images           7165 non-null   str  
 6   youtube_videos   7165 non-null   str  
 7   source_dataset   7165 non-null   str  
 8   cleaned_content  7140 non-null   str  
dtypes: str(9)
memory usage: 8.3 MB


In [21]:
forum_df['created_at'] = pd.to_datetime(forum_df['date'], utc=True)


In [22]:
forum_df = forum_df[forum_df['cleaned_content'].notna()].copy()
forum_df.info()


<class 'pandas.DataFrame'>
Index: 7140 entries, 0 to 7164
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype              
---  ------           --------------  -----              
 0   author           7140 non-null   str                
 1   content          7140 non-null   str                
 2   date             7140 non-null   str                
 3   post_id          7140 non-null   str                
 4   links            7140 non-null   str                
 5   images           7140 non-null   str                
 6   youtube_videos   7140 non-null   str                
 7   source_dataset   7140 non-null   str                
 8   cleaned_content  7140 non-null   str                
 9   created_at       7140 non-null   datetime64[us, UTC]
dtypes: datetime64[us, UTC](1), str(9)
memory usage: 8.4 MB


In [23]:
forum_df

,author,content,date,post_id,links,images,youtube_videos,source_dataset,cleaned_content,created_at
0,BrettS,339hp and 427ft lb torque\nhttps://www.carandd...,2019-05-21T11:02:14-0700,post-42418,['https://www.caranddriver.com/news/a27543113/...,['https://hips.hearstapps.com/hmg-prod.s3.amaz...,[],car_dyno,339hp and 427ft lb torque https://www.caranddr...,2019-05-21 18:02:14+00:00
2,Supra Turbo,Impressive. Can't wait to hear more about the ...,2019-05-21T11:11:58-0700,post-42421,[],[],[],car_dyno,Impressive. Can't wait to hear more about the ...,2019-05-21 18:11:58+00:00
3,justbake,At 15% drivetrain loss thats 398hp and 503 lb-...,2019-05-21T11:19:26-0700,post-42423,[],[],[],car_dyno,At 15% drivetrain loss thats 398hp and 503 lb-...,2019-05-21 18:19:26+00:00
4,A70TTR,"""it's so underpowered"" though",2019-05-21T11:54:42-0700,post-42431,[],[],[],car_dyno,"""it's so underpowered"" though",2019-05-21 18:54:42+00:00
5,Turbro,"A70TTR said:\n""it's so underpowered"" though\nC...",2019-05-21T12:23:26-0700,post-42442,['https://www.supramkv.com/goto/post?id=42431'],[],[],car_dyno,"A70TTR said: ""it's so underpowered"" though Cli...",2019-05-21 19:23:26+00:00
...,...,...,...,...,...,...,...,...,...,...
7160,TBK,General updates:\nTime to renew the registrati...,2026-01-31T22:54:19-0800,post-543464,[],['https://cdn.supramkv.com/attachments/151/151...,[],time_attack_build,General updates: Time to renew the registratio...,2026-02-01 06:54:19+00:00
7161,The Ginga Ninja,TBK said:\nAnybody have an experience working ...,2026-02-06T12:32:15-0800,post-545648,['https://www.supramkv.com/goto/post?id=543464'],[],[],time_attack_build,TBK said: Anybody have an experience working w...,2026-02-06 20:32:15+00:00
7162,Rensuhlo,TBK said:\nClick to expand...\nLooks...familia...,2026-02-06T13:34:37-0800,post-545666,['https://www.supramkv.com/goto/post?id=543464'],['https://cdn.supramkv.com/attachments/151/151...,[],time_attack_build,TBK said: Click to expand... Looks...familiar lol,2026-02-06 21:34:37+00:00
7163,TBK,The Ginga Ninja said:\ni have worked with Ryan...,2026-02-07T06:06:33-0800,post-545841,['https://www.supramkv.com/goto/post?id=545648...,['https://cdn.supramkv.com/attachments/151/151...,[],time_attack_build,The Ginga Ninja said: i have worked with Ryan ...,2026-02-07 14:06:33+00:00


In [24]:
for col in ['links', 'images', 'youtube_videos']:
    forum_df[col] = forum_df[col].apply(safe_literal_list)
    forum_df[f'{col}_count'] = forum_df[col].apply(len)


In [25]:
forum_df['evidence_id'] = forum_df.apply(lambda r: stable_id(r['source_dataset'], r['post_id'], prefix='ev_'), axis=1)
forum_df['vehicle_id'] = 'toyota_gr_supra_a90_b58'
forum_df['part_mentions'] = forum_df['cleaned_content'].apply(extract_part_mentions)
forum_df['mentioned_categories'] = forum_df['part_mentions'].apply(lambda x: ','.join(x))

In [26]:
forum_df

,author,content,date,post_id,links,images,youtube_videos,source_dataset,cleaned_content,created_at,links_count,images_count,youtube_videos_count,evidence_id,vehicle_id,part_mentions,mentioned_categories
0,BrettS,339hp and 427ft lb torque\nhttps://www.carandd...,2019-05-21T11:02:14-0700,post-42418,[],[],[],car_dyno,339hp and 427ft lb torque https://www.caranddr...,2019-05-21 18:02:14+00:00,0,0,0,ev_c49dff3453a6,toyota_gr_supra_a90_b58,"[aero, turbo, wheels_tires]","aero,turbo,wheels_tires"
2,Supra Turbo,Impressive. Can't wait to hear more about the ...,2019-05-21T11:11:58-0700,post-42421,[],[],[],car_dyno,Impressive. Can't wait to hear more about the ...,2019-05-21 18:11:58+00:00,0,0,0,ev_5548ac998c81,toyota_gr_supra_a90_b58,[],
3,justbake,At 15% drivetrain loss thats 398hp and 503 lb-...,2019-05-21T11:19:26-0700,post-42423,[],[],[],car_dyno,At 15% drivetrain loss thats 398hp and 503 lb-...,2019-05-21 18:19:26+00:00,0,0,0,ev_952edb11a903,toyota_gr_supra_a90_b58,[],
4,A70TTR,"""it's so underpowered"" though",2019-05-21T11:54:42-0700,post-42431,[],[],[],car_dyno,"""it's so underpowered"" though",2019-05-21 18:54:42+00:00,0,0,0,ev_6edb99acfcb9,toyota_gr_supra_a90_b58,[],
5,Turbro,"A70TTR said:\n""it's so underpowered"" though\nC...",2019-05-21T12:23:26-0700,post-42442,[],[],[],car_dyno,"A70TTR said: ""it's so underpowered"" though Cli...",2019-05-21 19:23:26+00:00,0,0,0,ev_93e2df779404,toyota_gr_supra_a90_b58,[],
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7160,TBK,General updates:\nTime to renew the registrati...,2026-01-31T22:54:19-0800,post-543464,[],[],[],time_attack_build,General updates: Time to renew the registratio...,2026-02-01 06:54:19+00:00,0,0,0,ev_1cddbfdb3e3e,toyota_gr_supra_a90_b58,"[aero, exhaust, suspension, wheels_tires]","aero,exhaust,suspension,wheels_tires"
7161,The Ginga Ninja,TBK said:\nAnybody have an experience working ...,2026-02-06T12:32:15-0800,post-545648,[],[],[],time_attack_build,TBK said: Anybody have an experience working w...,2026-02-06 20:32:15+00:00,0,0,0,ev_26c853dfafa4,toyota_gr_supra_a90_b58,"[aero, suspension]","aero,suspension"
7162,Rensuhlo,TBK said:\nClick to expand...\nLooks...familia...,2026-02-06T13:34:37-0800,post-545666,[],[],[],time_attack_build,TBK said: Click to expand... Looks...familiar lol,2026-02-06 21:34:37+00:00,0,0,0,ev_64dcd657582f,toyota_gr_supra_a90_b58,[],
7163,TBK,The Ginga Ninja said:\ni have worked with Ryan...,2026-02-07T06:06:33-0800,post-545841,[],[],[],time_attack_build,The Ginga Ninja said: i have worked with Ryan ...,2026-02-07 14:06:33+00:00,0,0,0,ev_d9fd978a0af9,toyota_gr_supra_a90_b58,"[aero, suspension]","aero,suspension"


A future analysis for Forum data would to determine what is a comment and what is a information providing post for each forum record

In [27]:
claims = forum_df['content'].apply(extract_numeric_claims).apply(pd.Series)
print(claims)

Empty DataFrame
Columns: []
Index: [0, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, ...]

[7140 rows x 0 columns]


In [30]:
forum_df['lap_time_seconds'] = np.where(forum_df['source_dataset'].isin(['lap_times', 'race_track', 'time_attack_build']), forum_df['cleaned_content'].apply(parse_lap_time_seconds), None)
forum_df['track_name'] = np.where(forum_df['source_dataset'].isin(['lap_times', 'race_track', 'time_attack_build']), forum_df['cleaned_content'].apply(extract_track), None)
forum_df['evidence_type'] = forum_df['source_dataset'].map({
    'car_dyno': 'dyno_evidence', 'exhaust_poll': 'user_preference_poll', 'lap_times': 'lap_time_evidence',
    'power_guide': 'power_build_guide', 'race_track': 'track_discussion', 'street_build_forum': 'street_build_log',
    'time_attack_build': 'time_attack_build_log'
})

In [31]:
forum_df.info()

<class 'pandas.DataFrame'>
Index: 7140 entries, 0 to 7164
Data columns (total 21 columns):
 #   Column                  Non-Null Count  Dtype              
---  ------                  --------------  -----              
 0   author                  7140 non-null   str                
 1   content                 7140 non-null   str                
 2   date                    7140 non-null   str                
 3   post_id                 7140 non-null   str                
 4   links                   7140 non-null   object             
 5   images                  7140 non-null   object             
 6   youtube_videos          7140 non-null   object             
 7   source_dataset          7140 non-null   str                
 8   cleaned_content         7140 non-null   str                
 9   created_at              7140 non-null   datetime64[us, UTC]
 10  links_count             7140 non-null   int64              
 11  images_count            7140 non-null   int64              

In [32]:
forum_df['evidence_quality_score'] = (
        30 + forum_df['links_count'].clip(0, 3) * 10 + forum_df['images_count'].clip(0, 2) * 10 +
        forum_df['youtube_videos_count'].clip(0, 1) * 10 + forum_df['part_mentions'].apply(lambda x: min(len(x), 3) * 5)
    ).clip(0, 100)

In [33]:
forum_df.describe()

,links_count,images_count,youtube_videos_count,evidence_quality_score
count,7140.0,7140.0,7140.0,7140.000000
mean,0.0,0.0,0.0,35.058824
std,0.0,0.0,0.0,4.845639
min,0.0,0.0,0.0,30.000000
25%,0.0,0.0,0.0,30.000000
50%,0.0,0.0,0.0,35.000000
75%,0.0,0.0,0.0,40.000000
max,0.0,0.0,0.0,45.000000


In [34]:
output_cols = [
        'evidence_id', 'vehicle_id', 'source_dataset', 'evidence_type', 'author', 'created_at', 'post_id',
        'cleaned_content', 'links', 'images', 'youtube_videos', 'links_count', 'images_count', 'youtube_videos_count',
        'mentioned_categories', 'lap_time_seconds', 'track_name', 'evidence_quality_score'
    ]
output_forum_df = forum_df[output_cols]


output_forum_df.to_csv(OUT_DIR / 'forum_evidence_clean.csv', index=False)